# **Chatbot using Transformers**


**1. Install & Import Libraries**

In [1]:
!pip install transformers torch

**Import Libraries**

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

**Load Pre-trained Model**

In [3]:
def simple_knowledge_response(text):
    text = text.lower()

    # Greetings
    if "hello" in text or "hi" in text:
        return "Hello Roshini! How can I assist you today?"

    # AI & ML
    elif "what is ai" in text:
        return "Artificial Intelligence is the ability of machines to perform tasks that require human intelligence."

    elif "what is machine learning" in text:
        return "Machine Learning is a subset of AI that allows systems to learn from data and improve automatically."

    elif "what is deep learning" in text:
        return "Deep Learning is a type of machine learning that uses neural networks with multiple layers."

    # Programming
    elif "who created python" in text:
        return "Python was created by :contentReference[oaicite:0]{index=0} in 1991."

    elif "what is python" in text:
        return "Python is a high-level, easy-to-learn programming language used for web development, AI, and data science."

    elif "what is java" in text:
        return "Java is a popular object-oriented programming language used for building applications."

    # Data Science
    elif "what is data science" in text:
        return "Data Science is the process of extracting insights from data using statistics, programming, and machine learning."

    elif "what is nlp" in text:
        return "Natural Language Processing is a field of AI that helps machines understand human language."

    # General Tech
    elif "what is cloud computing" in text:
        return "Cloud computing is the delivery of computing services like storage and servers over the internet."

    elif "what is chatbot" in text:
        return "A chatbot is a program that simulates human conversation using AI and NLP techniques."

    # Personal / Basic
    elif "how are you" in text:
        return "I'm doing great! How can I help you today?"

    elif "what is your name" in text:
        return "I am your AI assistant chatbot."

    elif "thank you" in text:
        return "You're welcome! Feel free to ask more questions."

    # Default
    return None

**Main Function**

In [6]:
chat_history_ids = None

print("Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.")

while True:

    user_input = input("You: ")

    # Exit condition
    if user_input.lower() in ['exit', 'quit']:
        print("Chatbot: Goodbye! 👋")
        break

    # 🔹 Step 1: Check knowledge-based response
    response = simple_knowledge_response(user_input)

    if response:
        print("Chatbot:", response)
        continue

    # 🔹 Step 2: Use transformer model

    # Add prompt for better responses
    prompt = f"User: {user_input}\nChatbot:"

    new_input_ids = tokenizer.encode(prompt + tokenizer.eos_token, return_tensors='pt')

    # Maintain chat history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=500,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.6,
        top_k=40,
        top_p=0.85,
        repetition_penalty=1.2
    )

    # Decode response
    bot_response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("Chatbot:", bot_response)

Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.
You: hello hi
Chatbot: Hello Roshini! How can I assist you today?
You: What is python?
Chatbot: Python is a high-level, easy-to-learn programming language used for web development, AI, and data science.
You: What is chatbot?
Chatbot: A chatbot is a program that simulates human conversation using AI and NLP techniques.
You: what is nlp?
Chatbot: Natural Language Processing is a field of AI that helps machines understand human language.
You: how are you ?
Chatbot: I'm doing great! How can I help you today?
You: what is your name ?
Chatbot: I am your AI assistant chatbot.
You: exit
Chatbot: Goodbye! 👋


* **Chatbot Pipeline**

User Input → Tokenization → Model Processing → Response Generation → Output → Loop

**Key Features**
* Used pre-trained transformer model (DialoGPT)
* Maintains conversation context
* Generates dynamic responses
* Supports continuous interaction
* Exit condition implemented

I built a chatbot using a pre-trained transformer model from Hugging Face. The model processes user input, maintains conversation history, and generates context-aware responses using text generation techniques.